In [1]:
import pandas as pd

df = pd.read_csv(
    '../data/emails.tsv',
    sep='\t',
    header=None,
    names=['label', 'text']
)

print(f"Total de emails: {len(df)}")

df.head(10)

Total de emails: 5572


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [2]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [3]:
phishing = df[df['label'] == 'spam']['text']
print("--- EJEMPLO PHISHING ---")
print(phishing.iloc[0])
print("\n--- EJEMPLO LEGÍTIMO ---")
print(df[df['label'] == 'ham']['text'].iloc[0])

--- EJEMPLO PHISHING ---
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

--- EJEMPLO LEGÍTIMO ---
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [4]:
df['longitud'] = df['text'].apply(len)
df.groupby('label')['longitud'].mean()

label
ham      71.482487
spam    138.670683
Name: longitud, dtype: float64

In [5]:
import nltk

from nltk.corpus import stopwords

print(stopwords.words('english')[:10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [7]:
import sys
sys.path.append('../src')
from preprocessor import limpiar_texto

# Probemos con un ejemplo real de phishing
ejemplo = "CONGRATULATIONS!! You've WON a FREE iPhone! Click http://win-now.com to claim NOW!!!"

print("ORIGINAL:")
print(ejemplo)
print("\nLIMPIO:")
print(limpiar_texto(ejemplo))


ORIGINAL:
CONGRATULATIONS!! You've WON a FREE iPhone! Click http://win-now.com to claim NOW!!!

LIMPIO:
congratulation youve free iphone click url claim


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from preprocessor import limpiar_texto, preparar_labels

# Limpiar todo el dataset
print("Limpiando textos...")
df['texto_limpio'] = df['text'].apply(limpiar_texto)
df['label_num'] = preparar_labels(df['label'])

print(f"Ejemplo limpio: {df['texto_limpio'].iloc[0]}")
print(f"Label: {df['label_num'].iloc[0]} ({df['label'].iloc[0]})")

Limpiando textos...
Ejemplo limpio: jurong point crazy available bugis great world buffet cine got amore wat
Label: 0 (ham)


In [9]:
# Crear el vectorizador
vectorizer = TfidfVectorizer(max_features=3000)

# Entrenar el vectorizador y transformar el texto
X = vectorizer.fit_transform(df['texto_limpio'])
y = df['label_num']

print(f"Forma de la matriz: {X.shape}")
print("Cada email es ahora un vector de 3000 números")

Forma de la matriz: (5572, 3000)
Cada email es ahora un vector de 3000 números


In [10]:
# Ver las palabras más "phishing" según TF-IDF
import pandas as pd
feature_names = vectorizer.get_feature_names_out()
top_phishing = df[df['label']=='spam']['texto_limpio']
X_phishing = vectorizer.transform(top_phishing)
scores = X_phishing.mean(axis=0).A1
top_words = pd.Series(scores, index=feature_names).sort_values(ascending=False).head(15)
print("Top palabras de phishing:")
print(top_words)

Top palabras de phishing:
numero          0.240294
call            0.070864
free            0.049857
txt             0.036493
mobile          0.035615
text            0.034543
claim           0.033425
url             0.030851
stop            0.030426
prize           0.029890
numeronumero    0.029254
reply           0.026762
service         0.024126
urgent          0.021513
new             0.021042
dtype: float64
